# Geometry Predictions — Explore Channel Geometry from a Trained KAN

This notebook loads the pre-computed geometry predictions NetCDF produced by
the `geometry_predictor.py` script and visualizes the results.

**To generate the dataset**, run the Hydra script first:

```bash
python scripts/geometry_predictor.py --config-name=merit_geometry_config
```

The script uses a trained KAN checkpoint to predict channel parameters (n, p, q)
for all CONUS reaches, then batches through a water year of real accumulated
discharge to compute trapezoidal geometry statistics (min/max/median/mean) for
depth, top width, and discharge.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
xr.open_dataset("merit_geometry_predictions.nc")

## 1. Load the geometry predictions NetCDF

Update the path below to point to the output of `geometry_predictor.py`.

In [ ]:
PREDICTIONS_PATH = Path("merit_geometry_predictions.nc")

ds = xr.open_dataset(PREDICTIONS_PATH)
print(f"Reaches:    {ds.attrs['n_reaches']:,}")
print(f"Days:       {ds.attrs['n_days']}")
print(f"Water year: {ds.attrs['water_year']}")
ds

## 2. Distributions of learned KAN parameters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

params = {
    "n (Manning's roughness)": ds["n"].values,
    "q_spatial (width-depth exponent)": ds["q_spatial"].values,
    "p_spatial (width coefficient)": ds["p_spatial"].values,
}

for ax, (label, values) in zip(axes, params.items(), strict=False):
    valid = values[~np.isnan(values)]
    ax.hist(valid, bins=50, edgecolor="black", linewidth=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.axvline(np.median(valid), color="red", linestyle="--", label=f"median={np.median(valid):.3f}")
    ax.legend()

fig.suptitle("Learned KAN Parameters across CONUS Reaches", fontsize=13)
fig.tight_layout()
plt.show()

## 3. Depth and top width statistics across the water year

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for row, var in enumerate(["depth", "top_width"]):
    for col, stat in enumerate(["median", "min", "max"]):
        ax = axes[row, col]
        key = f"{var}_{stat}"
        vals = ds[key].values
        valid = vals[~np.isnan(vals)]
        ax.hist(valid, bins=80, edgecolor="black", linewidth=0.2)
        ax.set_xlabel(f"{var} {stat} (m)")
        ax.set_ylabel("Count")
        ax.set_title(key)

fig.suptitle("Depth and Top Width Statistics (WY2000)", fontsize=13)
fig.tight_layout()
plt.show()

## 4. Geometry vs. discharge (median values)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

q_med = ds["discharge_median"].values
mask = q_med > 0

ax = axes[0]
ax.scatter(q_med[mask], ds["depth_median"].values[mask], s=0.5, alpha=0.2, rasterized=True)
ax.set_xlabel("Discharge median (m³/s)")
ax.set_ylabel("Depth median (m)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Depth vs. Discharge")

ax = axes[1]
ax.scatter(q_med[mask], ds["top_width_median"].values[mask], s=0.5, alpha=0.2, rasterized=True)
ax.set_xlabel("Discharge median (m³/s)")
ax.set_ylabel("Top Width median (m)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Top Width vs. Discharge")

fig.suptitle("Predicted Channel Geometry vs. Discharge (WY2000 medians)", fontsize=13)
fig.tight_layout()
plt.show()

## 5. Leopold & Maddock: Width vs. Depth

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

depth = ds["depth_median"].values
width = ds["top_width_median"].values
mask = ~(np.isnan(depth) | np.isnan(width)) & (depth > 0) & (width > 0)

ax.scatter(depth[mask], width[mask], s=0.5, alpha=0.2, rasterized=True, label="Predicted")
ax.set_xlabel("Depth (m)")
ax.set_ylabel("Top Width (m)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Leopold & Maddock: Width vs. Depth (median)")

# Overlay median power law: W = p_median * D^q_median
p_med = np.nanmedian(ds["p_spatial"].values)
q_med = np.nanmedian(ds["q_spatial"].values)
d_range = np.logspace(np.log10(np.nanmin(depth[mask])), np.log10(np.nanmax(depth[mask])), 100)
ax.plot(
    d_range, p_med * d_range**q_med, "r-", linewidth=2, label=f"W = {p_med:.1f} D^{q_med:.2f} (median params)"
)
ax.legend()

fig.tight_layout()
plt.show()

## 6. Summary statistics

In [ ]:
summary = {}
for var in ds.data_vars:
    vals = ds[var].values
    summary[var] = {
        "min": np.nanmin(vals),
        "p10": np.nanpercentile(vals, 10),
        "median": np.nanmedian(vals),
        "mean": np.nanmean(vals),
        "p90": np.nanpercentile(vals, 90),
        "max": np.nanmax(vals),
    }

pd.DataFrame(summary).T.round(4)